<a href="https://colab.research.google.com/github/KCDAS35/WEB/blob/main/demo/omnivoice_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OmniVoice-TTS Colab — T4 Quickstart

**Features:** Zero-shot voice cloning · Cross-lingual TTS (600+ languages) · Voice Design · Batch inference

> **Requirements:** T4 GPU runtime · Model: `k2-fsa/OmniVoice` (auto-downloaded on first run)

## Step 1: GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU detected: {gpu} ({vram:.1f} GB VRAM)')
    if vram < 4:
        print('⚠️  WARNING: Less than 4 GB VRAM. Short outputs only — longer text may fail.')
    elif vram >= 8:
        print('✅ 8+ GB VRAM — full feature set available including longer outputs.')
else:
    print("""
    ⚠️ WARNING: No GPU detected.

    OmniVoice requires a GPU. To change runtime:
        1. Runtime → Change runtime type
        2. Select T4 GPU
        3. Save and reconnect
    """)

## Step 2: Install OmniVoice

In [ ]:
# Install OmniVoice (Colab already has compatible PyTorch+CUDA)
!pip install -q omnivoice
print('✅ OmniVoice installed')

# Install cloudflared for public URL tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
print('✅ Cloudflared ready')

## Step 3: Launch OmniVoice Demo

The model (`k2-fsa/OmniVoice`) downloads automatically on first run (~few GB).

The Gradio interface includes:
- **Zero-Shot Voice Cloning** — upload any audio sample (3 seconds is enough)
- **Cross-Lingual TTS** — clone a voice in one language, speak in another (600+ languages)
- **Voice Design** — build a voice from scratch: Gender, Age, Accent, Style
- **Batch Inference** — process multiple texts at once

> **Tip:** Leave language on **Auto** for best cross-lingual results.

In [ ]:
import subprocess, re, threading, time

demo = subprocess.Popen(
    'omnivoice-demo --ip 0.0.0.0 --port 8001',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
cf = subprocess.Popen(
    '/content/cloudflared tunnel --url http://localhost:8001 --no-autoupdate',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

url_pattern = re.compile(r'(https://[a-z0-9-]+\.trycloudflare\.com)')

def watch_demo():
    for line in demo.stdout:
        print('[omnivoice]', line.strip())

def watch_cf():
    for line in cf.stdout:
        m = url_pattern.search(line)
        if m:
            print(f'\n✅ OmniVoice Public URL: {m.group(1)}')
            print('Share this link — works on any device until the Colab session ends.\n')

threading.Thread(target=watch_demo, daemon=True).start()
threading.Thread(target=watch_cf, daemon=True).start()

print('Launching OmniVoice... wait ~30 seconds for the public URL to appear.')
while True:
    time.sleep(1)

## Step 4: Long-Form TTS — Paste Millions of Words

Use this cell to synthesize **long texts** (books, articles, scripts, entire documents).
The text is automatically split into ~400-word chunks at sentence boundaries,
each chunk is synthesized separately, then all audio segments are joined into a single file.

**Supports:** voice cloning (`REF_AUDIO`) · voice design (`INSTRUCT`) · any length of text

> Run **Step 3** first so the model is loaded, then come back here and run this cell.

In [ ]:
import re, os, subprocess
import numpy as np
import soundfile as sf

# ── CONFIG ──────────────────────────────────────────────────────────────────
LONG_TEXT = """
Paste your full text here. No limit — 1,000 words, 20,000 words, 100,000 words,
whatever you need. Sentence breaks are detected automatically.
""".strip()

REF_AUDIO  = None   # Voice clone: path to audio file, e.g. '/content/voice.wav'
INSTRUCT   = None   # Voice design: e.g. 'female, warm, Latin American Spanish accent'
MAX_WORDS  = 400    # Words per chunk (200–500 recommended; lower = safer on low VRAM)
OUTPUT     = '/content/output_longform.wav'
# ────────────────────────────────────────────────────────────────────────────


def chunk_text(text, max_words=400):
    """Split text into chunks at sentence boundaries."""
    text = re.sub(r'\s+', ' ', text.strip())
    sentences = re.split(r'(?<=[.!?\u2026])\s+', text)
    chunks, current, count = [], [], 0
    for sent in sentences:
        w = len(sent.split())
        if count + w > max_words and current:
            chunks.append(' '.join(current))
            current, count = [sent], w
        else:
            current.append(sent)
            count += w
    if current:
        chunks.append(' '.join(current))
    return chunks


def synthesize_long(text, ref_audio=None, instruct=None,
                    output=OUTPUT, max_words=MAX_WORDS):
    """Synthesize arbitrarily long text via chunked OmniVoice inference."""
    chunks = chunk_text(text, max_words)
    total  = len(text.split())
    print(f'\U0001f4dd {total:,} words -> {len(chunks)} chunks (max {max_words} words each)\n')

    wav_files = []
    for i, chunk in enumerate(chunks):
        out = f'/content/_chunk_{i:04d}.wav'
        print(f'  [{i+1}/{len(chunks)}] {len(chunk.split())} words ... ', end='', flush=True)
        cmd = ['omnivoice-infer', '--model', 'k2-fsa/OmniVoice',
               '--text', chunk, '--output', out]
        if ref_audio:
            cmd += ['--ref_audio', ref_audio]
        if instruct:
            cmd += ['--instruct', instruct]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'FAILED \u26a0\ufe0f')
            print(f'    {r.stderr[:300]}')
        else:
            print('done \u2705')
            wav_files.append(out)

    if not wav_files:
        print('\u274c Nothing synthesized — check errors above.')
        return None

    print(f'\n\U0001f517 Joining {len(wav_files)} segments ...')
    arrays, sr = [], None
    for f in wav_files:
        data, rate = sf.read(f)
        sr = rate
        arrays.append(data)
        os.remove(f)

    sf.write(output, np.concatenate(arrays), sr)
    mins = total / 150  # ~150 words per minute spoken
    print(f'\u2705 Saved: {output}')
    print(f'   Words: {total:,} | Estimated audio length: ~{mins:.1f} min')
    return output


synthesize_long(LONG_TEXT, ref_audio=REF_AUDIO, instruct=INSTRUCT)


## Step 5: Relay API — Serve this GPU to your Local OmniVoice App

Instead of running inference on your laptop, your **local OmniVoice app**
(`apps/omnivoice/app.py`) can POST each chunk here and get back a WAV.
The model is loaded once on this Colab T4, stays hot, and processes
chunks at GPU speed.

This cell runs the **same `apps/omnivoice/relay.py`** that powers the
RunPod launcher (`apps/omnivoice/runpod_start.sh`) — one relay
implementation, three places it can live (Colab T4, RunPod, any GPU box).

**How to use:**
1. Run this cell — wait for the `Relay URL:` line.
2. Copy that `https://xxxxx.trycloudflare.com` URL.
3. In your local OmniVoice app, pick **Remote — Colab relay** as the
   backend and paste the URL.

> Do **not** run Step 3 and Step 5 at the same time — they both load
> the model and will fight for VRAM.

In [ ]:
# Clone WEB repo (if not already present) and run the shared relay.
import os, subprocess, re, threading, time

REPO_DIR = '/content/WEB'
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1',
                           'https://github.com/KCDAS35/WEB.git', REPO_DIR])

subprocess.check_call(['pip', 'install', '-q', 'fastapi', 'uvicorn',
                       'python-multipart'])

relay = subprocess.Popen(
    ['python', f'{REPO_DIR}/apps/omnivoice/relay.py', '--port', '8002'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

def watch_relay():
    for line in relay.stdout:
        print('[relay]', line.rstrip())
threading.Thread(target=watch_relay, daemon=True).start()

time.sleep(5)

cf = subprocess.Popen(
    '/content/cloudflared tunnel --url http://localhost:8002 --no-autoupdate',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1)

url_re = re.compile(r'(https://[a-z0-9-]+\.trycloudflare\.com)')
def watch_cf():
    for line in cf.stdout:
        m = url_re.search(line)
        if m:
            print(f'\n\u2705 Relay URL: {m.group(1)}')
            print('Paste this into your local OmniVoice app (backend: Remote).\n')
threading.Thread(target=watch_cf, daemon=True).start()

print('Relay starting... URL appears in ~15\u201330 s. Keep this cell running.')
while True:
    time.sleep(2)


## Tips & Feature Guide

### Voice Cloning (CLI alternative)
```bash
omnivoice-infer --model k2-fsa/OmniVoice \
    --text "Bienvenido a nuestra clínica dental" \
    --ref_audio ref.wav \
    --output output.wav
```

### Voice Design (CLI alternative)
```bash
omnivoice-infer --model k2-fsa/OmniVoice \
    --text "Hello, welcome to our service" \
    --instruct "female, warm, Latin American Spanish accent" \
    --output output.wav
```

### Cross-Lingual TTS (Perfect for Lima Clients)
- Upload an English voice sample → output Spanish speech in that same voice
- OmniVoice supports 600+ languages natively

### Batch Processing
```bash
omnivoice-infer-batch --model k2-fsa/OmniVoice --input batch.json --output-dir ./outputs
```